In [ ]:
# Imports

import chromadb
from sentence_transformers import SentenceTransformer
import pandas as pd
import os
from dotenv import load_dotenv
from openai import OpenAI
import json
print("Imports done")

In [ ]:
# Chroma DB

chroma_client = chromadb.PersistentClient(
    path="../data/snomed/chroma_db"
)
collection = chroma_client.get_collection(name="snomed_concepts")
print(f"Collection loaded: {collection.count()} concepts")

In [ ]:
# Embedding model

model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded")

In [ ]:
# OpenAI client

load_dotenv("../.env")
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
print("OpenAI client ready")

In [ ]:
# Gold standard

gold_standard = pd.read_csv(
    "../data/gold_standards/nhsd-primary-care-domain-refsets-dmtype2_cod-20250912.csv",
    dtype={'code': str}
)
print(f"Gold standard loaded: {len(gold_standard)} codes")
print(gold_standard.head(23).to_string())

In [ ]:
def retrieve_snomed_concepts(query, n_results=400):
    """
    Retrieve semantically similar SNOMED-CT concepts.
    n_results=400 ensures full retrieval coverage for the largest
    gold standard codelist (IHD, 323 codes) plus a buffer.
    """
    query_embedding = model.encode([query])
    
    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=n_results
    )
    
    retrieved = []
    for code, term in zip(results['ids'][0], results['documents'][0]):
        retrieved.append({'code': code, 'term': term})
    
    return retrieved

In [ ]:
# Tweaking retieval call

retrieved_concepts = retrieve_snomed_concepts(
    "Type 2 Diabetes Mellitus", 
    n_results=400
)
retrieved_codes = set([c['code'] for c in retrieved_concepts])
gold_codes = set(gold_standard['code'])
coverage = gold_codes & retrieved_codes
print(f"Gold standard codes retrieved: {len(coverage)}/{len(gold_codes)}")
print(f"Coverage: {len(coverage)/len(gold_codes)*100:.1f}%")

In [ ]:
# Examining the missed codes

# Get retrieved codes from ChromaDB at n=400
retrieved_concepts = retrieve_snomed_concepts("Type 2 Diabetes Mellitus", n_results=400)

# Extract just the codes into a set
retrieved_codes = set([c['code'] for c in retrieved_concepts])

# Extract gold standard codes into a set
gold_codes = set(gold_standard['code'])

# Find missed codes — in gold standard but NOT in retrieved
missed_codes = gold_codes - retrieved_codes

# Filter gold standard dataframe to show only missed codes
missed = gold_standard[gold_standard['code'].isin(missed_codes)]

print(f"Missed codes: {len(missed_codes)}/23")
print(missed[['code', 'term']])  # fill in the term/description column name from your dataframe

In [ ]:
import chromadb
client = chromadb.PersistentClient(path="../data/snomed/chroma_db")
print(dir(client))

In [ ]:
collection = client.get_collection(name="snomed_concepts")
help(collection.get)

In [ ]:
os.getcwd()

In [ ]:
help(collection.get)